# 3.0 · 인사이트와 최종 후보 (행정동 단위)

**목표**: `2.0-modeling.ipynb`는 자치구 단위(25개)에서 "왜 어떤 자치구는 카페 매출이 높은가"를 설명했습니다. 이 노트북은 한 단계 더 들어가서, 실제로 개업을 고려할 만한 **행정동(동네) 단위 후보지**를 420개 중에서 좁히고, 그 과정에서 나온 지표들을 **상관관계와 인과관계를 구분**해서 실행 가능한 비즈니스 인사이트로 정리합니다.

행정동 단위 표에는 자치구 단위에 있던 배후 수요 변수(직장인구·상주인구 등)가 없는 대신, **매출, 성장률, 시간대별 매출 비중(컨셉), 점포 수**가 있습니다. 그래서 이 노트북에서는 회귀모델이 아니라 **조건을 순서대로 좁혀가는 방식**으로 후보를 선정합니다.

## STEP 1. 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

sns.set_style("whitegrid")
# Windows에서 한글이 깨지지 않도록 폰트를 지정합니다. (다른 컴퓨터에서 실행 시 설치된 한글 폰트로 바꿔주세요)
# 주의: sns.set_style()이 font.family를 다시 초기화하므로, 반드시 그 다음에 폰트를 지정해야 합니다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# 실행 전 확인: 이 변수만 본인 컴퓨터의 프로젝트 경로로 바꿔주세요.
# ------------------------------------------------------------
프로젝트_루트 = r"C:\Users\seokho\Desktop\엘리스\ABA_1st_proj_수정"

행정동_경로 = 프로젝트_루트 + r"\data\processed\서울_최종후보_행정동.csv"
자치구_모델결과_경로 = 프로젝트_루트 + r"\data\processed\서울_자치구_모델링결과.csv"

행정동 = pd.read_csv(행정동_경로, encoding="utf-8-sig")
자치구_모델결과 = pd.read_csv(자치구_모델결과_경로, encoding="utf-8-sig")

print("행정동 표본:", len(행정동), "개")
행정동[["행정동_코드_명", "점포_수", "카페1개당_평균매출_억원", "성장률", "컨셉_적합도"]].head(3)

## STEP 2. 표본 안정성 필터 — 점포 수 기준

`1.0-eda.ipynb`에서 점포 수가 5개 미만인 행정동은 매출 평균이 튀는 값 하나에 크게 흔들릴 수 있다고 확인했었죠. 최종 후보를 뽑을 때는 이 문제를 더 엄격하게 봐야 합니다 — 개업을 진지하게 고려할 동네라면 "카페가 몇 개나 있고, 그 동네에서 카페 장사가 어떤지"에 대한 근거가 충분해야 하니까요. 그래서 **점포 수 20개 이상**인 행정동만 "안정적 표본"으로 취급합니다.

In [ ]:
안정표본 = 행정동[행정동["점포_수"] >= 20].copy()
print(f"전체 {len(행정동)}개 행정동 중, 점포 20개 이상(안정적 표본) {len(안정표본)}개")
print(f"제외된 {len(행정동) - len(안정표본)}개는 카페 몇 개만으로 동네 전체를 판단하기엔 근거가 부족해 최종 후보에서 제외합니다.")

## STEP 3. 매출 Top20 + 역성장 제외

순간의 매출 스냅샷만 보면 "지금은 잘 나가지만 꺾이고 있는 곳"을 걸러낼 수 없습니다. 그래서 안정적 표본 중 카페1개당 평균매출 Top20을 뽑고, 그 안에서 최근 성장률(2025년 1분기 대비 4분기)이 마이너스인 곳은 제외합니다.

In [ ]:
Top20 = 안정표본.sort_values("카페1개당_평균매출_억원", ascending=False).head(20)

print("=== 안정 표본 매출 Top20 (성장률 포함) ===")
print(Top20[["행정동_코드_명", "점포_수", "카페1개당_평균매출_억원", "성장률"]].to_string(index=False))

역성장_제외 = Top20[Top20["성장률"] < 0]
print(f"\n이 중 역성장(성장률 < 0) 중인 곳 {len(역성장_제외)}개는 제외합니다:")
print(역성장_제외[["행정동_코드_명", "카페1개당_평균매출_억원", "성장률"]].to_string(index=False))

후보 = Top20[Top20["성장률"] >= 0].copy()
print(f"\n최종 후보군: {len(후보)}개")

매출 Top20 안에도 명동(0.95억원, 성장률 -5.3%), 자양3동(-6.6%), 위례동(-22.1%), 역삼1동(-1.6%), 방배본동(-5.3%)처럼 **매출은 상위권이지만 최근 분기 흐름이 꺾이고 있는 곳**이 여럿 있습니다. 이런 곳은 "지금 스냅샷"만 보고 들어가면 위험할 수 있어 제외하고, 남은 **15곳**을 후보군으로 삼습니다.

## STEP 4. 컨셉_적합도(합계) 지표의 한계

지금까지 써온 `컨셉_적합도`는 아침(06~11시)+점심(11~14시)+저녁(17~21시) 매출 비중을 **그냥 더한 값**입니다. 문제는 이 방식이 "세 시간대 비중의 총량"만 잴 뿐, "세 시간대에 고르게 퍼져 있는지"는 재지 못한다는 점입니다. 실제 사례로 확인해봅니다.

In [ ]:
사례 = 후보[후보["행정동_코드_명"].isin(["소공동", "여의동"])]
print(사례[["행정동_코드_명", "아침_비중", "점심_비중", "저녁_비중", "컨셉_적합도"]].to_string(index=False))

여의동은 점심 시간대에만 45.6%가 몰려 있는데도(아침 23.2%, 저녁 10.2%), 세 비중의 합(컨셉_적합도=79.0)은 후보군 15곳 중 1위로 나옵니다. "아침·점심·저녁 골고루 파는 카페"라는 원래 취지와는 거리가 있는데도, 단순 합계라는 계산 방식 때문에 1위처럼 보이는 거예요. 이 문제를 바로잡기 위해 "고르게 분산되어 있는 정도"를 직접 재는 지표가 필요합니다.

## STEP 5. 균형점수(엔트로피 기반) 계산

**엔트로피**는 원래 정보이론에서 "값들이 얼마나 고르게 퍼져 있는지"를 재는 지표인데, 여기서는 쉽게 "아침·점심·저녁 매출이 세 시간대에 얼마나 골고루 나뉘어 있는가"를 0~100 점수로 바꾼 것이라고 생각하면 됩니다. 세 시간대가 정확히 1/3씩 똑같으면 100점(가장 균형 잡힘), 한 시간대에 매출이 완전히 몰리면 0점(가장 쏠림)에 가까워집니다.

계산 방법은: 세 비중을 합이 1이 되도록 정규화한 뒤(예: 23.2/45.6/10.2 → 이 셋의 비율), 정보이론 공식(−Σp·log₂p)으로 "퍼짐 정도"를 구하고, 최댓값(완전 균등일 때)으로 나눠서 0~100 점수로 바꿉니다. 공식 자체보다 **"합계가 아니라 균형(고르게 퍼진 정도)을 잰다"는 방향성**이 중요합니다.

In [ ]:
def 균형점수_엔트로피(아침, 점심, 저녁):
    값 = np.array([아침, 점심, 저녁], dtype=float)
    합계 = 값.sum()
    if 합계 == 0:
        return 0.0
    확률 = 값 / 합계  # 세 시간대 비중을 "확률"처럼 취급 (합이 1이 되도록)
    엔트로피 = -sum(p * math.log2(p) for p in 확률 if p > 0) / math.log2(3)  # 0~1로 정규화
    return round(엔트로피 * 100, 1)

후보["균형점수"] = [
    균형점수_엔트로피(a, l, d)
    for a, l, d in zip(후보["아침_비중"], 후보["점심_비중"], 후보["저녁_비중"])
]

후보[["행정동_코드_명", "아침_비중", "점심_비중", "저녁_비중", "컨셉_적합도", "균형점수"]].sort_values("균형점수", ascending=False)

## STEP 6. 기존 방식(합계) vs 균형점수 — 순위가 뒤바뀌는 곳 확인

In [ ]:
후보["순위_기존합계"] = 후보["컨셉_적합도"].rank(ascending=False, method="min").astype(int)
후보["순위_균형점수"] = 후보["균형점수"].rank(ascending=False, method="min").astype(int)
후보["순위변화"] = 후보["순위_기존합계"] - 후보["순위_균형점수"]

비교표 = 후보[["행정동_코드_명", "컨셉_적합도", "균형점수", "순위_기존합계", "순위_균형점수", "순위변화"]].sort_values("순위_기존합계")
비교표

여의동은 컨셉_적합도(합계) 기준으로는 후보군 15곳 중 1위였지만, 균형점수 기준으로는 15곳 중 하위권으로 순위가 크게 떨어집니다. "세 시간대에 고르게 판다"는 원래 취지에 비춰보면, 균형점수가 이 취지를 훨씬 더 정확하게 반영하는 지표라고 판단해 **최종 후보 선정에는 균형점수를 기준으로 씁니다.**

## STEP 7. 민감도 분석 — 균형점수 1위가 우연이 아닌지 확인

후보군 15곳 안에서 균형점수만 보면 1위가 나오긴 하지만, 표본(점포 수)이 작은 곳은 우연히 고르게 나왔을 가능성(노이즈)을 배제할 수 없습니다. 그래서 점포 수 기준을 20개 → 30 → 40 → 50 → 75 → 100개로 점점 엄격하게 올려가면서, "그 안에서 균형점수 1위가 누구인가"가 안정적으로 유지되는지 확인합니다.

In [ ]:
print("=== 표본 기준을 올려가며 균형점수 1위가 누구로 바뀌는지 ===")
민감도_결과 = []
for 기준 in [20, 30, 40, 50, 75, 100]:
    부분집합 = 후보[후보["점포_수"] >= 기준]
    if len(부분집합) == 0:
        print(f"점포수 >= {기준}: 남는 후보 없음")
        continue
    선두 = 부분집합.sort_values("균형점수", ascending=False).iloc[0]
    민감도_결과.append(선두["행정동_코드_명"])
    print(
        f"점포수 >= {기준:>3} (후보 {len(부분집합):>2}곳 남음) -> 1위: {선두['행정동_코드_명']} "
        f"(균형점수 {선두['균형점수']}, 점포수 {int(선두['점포_수'])}, "
        f"매출 {선두['카페1개당_평균매출_억원']}억원, 성장률 {선두['성장률']}%)"
    )

점포 수 20개 기준에서는 수서동(균형점수 97.2, 점포수 21개)이 1위지만, 딱 21개로 기준을 겨우 넘긴 수준이라 우연히 고르게 나왔을 가능성이 있습니다. 기준을 30~50개로 올리면 **서초4동**이 계속 1위를 지키고, 75~100개로 더 올리면 대치4동으로 바뀝니다. 30~50개 구간에서 일관되게 1위를 지킨 서초4동을 "표본이 작아서 생긴 우연이 아니라 실제로 안정적인 균형 매출을 보이는 곳"으로 판단해 최종 후보로 채택합니다(수서동은 20개 구간에서만 반짝 1위였던 것으로 보고 제외).

## STEP 8. 최종 후보 3곳 확정

In [ ]:
매출_1_2위 = 후보.sort_values("카페1개당_평균매출_억원", ascending=False).head(2)
균형_안정승자 = 후보[후보["행정동_코드_명"] == "서초4동"]

최종3곳 = pd.concat([매출_1_2위, 균형_안정승자]).drop_duplicates(subset="행정동_코드_명")
최종3곳[["행정동_코드_명", "자치구_코드", "점포_수", "카페1개당_평균매출_억원", "성장률", "컨셉_적합도", "균형점수"]]

**최종 후보 3곳**: 소공동(매출 1위) · 잠실2동(매출 2위) · 서초4동(균형점수 안정적 1위, 민감도 분석으로 검증). 원래 컨셉적합도(합계) 1위였던 여의동은 STEP6~7에서 확인했듯 실제로는 점심 시간대에 쏠린 곳이라 균형 기준으로는 후보군 하위권이라 최종 3곳에서 제외했습니다.

## STEP 8-1. 자치구 단위 모델과 교차 확인

`2.0-modeling.ipynb`의 자치구 단위 모델(잔차 분석)로 돌아가서, 이 3개 후보 행정동이 속한 자치구가 그 모델에서 어떻게 나타났는지 확인해봅니다. 자치구 모델이 "직장인구·가구수·집객시설"만으로 예측한 값보다 실제 매출이 얼마나 높았는지(잔차)를 보면, 행정동 단위 선정과 자치구 단위 모델이 서로 다른 각도에서도 앞뒤가 맞는지 교차 검증할 수 있습니다.

In [ ]:
자치구_코드_대응 = {"소공동": 11140, "잠실2동": 11710, "서초4동": 11650}

교차확인 = 자치구_모델결과[자치구_모델결과["자치구_코드"].isin(자치구_코드_대응.values())][
    ["자치구_코드", "자치구_코드_명", "카페1개당_평균매출", "예측_카페1개당매출", "잔차"]
].copy()
교차확인["잔차_순위(25개_자치구_중)"] = 자치구_모델결과["잔차"].rank(ascending=False, method="min")[교차확인.index].astype(int)
교차확인.insert(0, "관련_최종후보", 교차확인["자치구_코드"].map({v: k for k, v in 자치구_코드_대응.items()}))
교차확인

소공동이 속한 **중구**와 잠실2동이 속한 **송파구**는 25개 자치구 중 잔차 순위 2위·7위로, 자치구 모델(직장인구·가구수·집객시설)이 예측한 것보다 실제 매출이 더 높게 나오는 지역입니다 — 모델에 안 들어간 다른 강점(관광·유동인구 등)이 있을 가능성이 높고, 이는 소공동·잠실2동이 행정동 단위에서도 매출 1·2위로 뽑힌 것과 방향이 일치합니다.

반면 서초4동이 속한 **서초구**는 잔차 순위가 25개 자치구 중 17위로 중하위권입니다(자치구 모델이 예측한 값보다 실제 매출이 오히려 낮은 편). 그런데도 서초4동이 행정동 단위에서 후보로 뽑힌 이유는 매출 규모가 아니라 **균형점수(시간대 분산)** 때문이라는 걸 이 교차 확인으로 다시 한번 분명히 할 수 있습니다 — 즉 서초4동은 "자치구 전체가 카페 장사가 유독 잘되는 동네"라서가 아니라, "그 안에서 시간대에 관계없이 안정적으로 팔리는 동네"라서 선택된 것입니다. 두 선정 기준(매출 규모 vs 안정성)이 서로 다른 이유로 후보에 오른 것이라는 점을 인사이트로 남깁니다.

## STEP 9. 상관관계 vs 인과관계 — 모델이 말해주지 않는 것

`2.0-modeling.ipynb`의 최종 모델은 `카페1개당_평균매출 ~ log(총_직장_인구_수) + 총_가구_수 + 집객시설_수`였고, R²=0.85(LOOCV 0.78)로 꽤 잘 설명했습니다. 그런데 이 결과를 "직장인구를 늘리면 카페 매출이 오른다"처럼 **인과관계로 확대 해석하면 안 됩니다.** 이유는 세 가지입니다.

1. **역의 인과 가능성**: 카페 매출이 높은 상권이라서 오히려 직장·상업시설이 더 들어왔을 수도 있습니다(원인과 결과가 뒤바뀔 수 있음).
2. **공통 원인(교란변수) 가능성**: "상권이 얼마나 성숙했는가"·"지가·임대료 수준" 같이 우리가 측정하지 못한 제3의 요인이 직장인구와 카페 매출을 동시에 밀어올렸을 수 있습니다.
3. **관측 데이터의 한계**: 이 데이터는 실험이 아니라 이미 존재하는 25개 자치구를 있는 그대로 관찰한 것이라, "만약 이 지역에 직장인구를 인위적으로 늘리면 매출이 오를까"라는 개입(intervention) 질문에는 답할 수 없습니다.

그래서 이 프로젝트의 결론은 "직장인구가 매출을 발생시킨다"가 아니라, **"직장인구·주거 성격·집객시설이라는 세 가지 특징을 함께 가진 지역에서 카페 매출이 높게 관찰된다"**는 상관관계 수준의 설명입니다. 실행 가능한 인사이트로 쓰기에는 충분하지만(그런 특징을 가진 동네를 찾으면 됨), "카페를 열면 매출이 오른다"는 식의 확대 해석은 하지 않습니다.

## STEP 10. 종합 결론

이 프로젝트는 실제로 서로 다른 세 가지 질문을 던졌고, 답이 서로 다릅니다. 셋을 섞으면 결론이 흐려집니다.

| 질문 | 답 | 근거 |
|---|---|---|
| 서울에서 카페 매출이 높은 자치구의 공통 특징이 있는가 | 있다 — 직장인구가 많고, 가구 수(주거지 성격)가 적고, 집객시설이 많은 자치구 | 자치구 모델 LOOCV R²=0.78, VIF 전부 3 미만, 세 계수 모두 p<0.05 (`2.0-modeling` STEP4·4-부록) |
| 이 자치구 단위 특징만으로 최적의 "행정동"을 바로 찍을 수 있는가 | 아니다 — 자치구 수준 설명력이 행정동 단위 후보 선정까지 그대로 이어지지 않는다 | 서초4동은 자치구(서초구) 잔차 순위가 25개 중 17위로 중하위권인데도, 행정동 단위 균형점수 기준으로는 최종 후보에 오름 (STEP 8-1) |
| 지금 매출이 높은 지역이 앞으로도 계속 좋은 입지인가 | 아니다 — 매출 상위권에도 최근 역성장 중인 곳이 다수 | 안정 표본 매출 Top20 중 5곳(명동·자양3동·위례동·역삼1동·방배본동)이 역성장으로 최종 후보에서 제외 (STEP 3) |
| "직장인구가 많으면 카페 매출이 오른다"는 인과관계로 볼 수 있는가 | 알 수 없다 — 상관관계 수준 | 단면 관측 데이터라 역인과·교란변수 가능성을 배제할 수 없음 (STEP 9) |

**이 결론은 얼마나 견고한가**: 분석 선택을 바꿔가며 점검했고 결론이 유지됩니다. ①변수 선택 — 후보를 관련성 있는 변수 8개로 제한했고, 4번째 변수부터는 LOOCV가 개선되지 않는 것을 확인했으며, 난수 50개 실험으로 선택편향 위험이 완전히 0은 아니라는 점까지 정직하게 확인했습니다(`2.0-modeling` STEP3-부록). ②모델 가정 — VIF 전부 3 미만, 계수 유의성 p<0.05로 다중공선성·유의성 문제가 없음을 확인했습니다(STEP4-부록). ③대안 모델 — 랜덤포레스트로 바꿔도 LOOCV가 오히려 나빠져(0.78→0.41) 선형회귀 선택이 정당함을 확인했습니다(STEP6). ④후보 선정 방식 — 컨셉_적합도(합계) 대신 균형점수(엔트로피)로 지표를 바꿔도, 점포수 기준을 20~100개로 올려가며 봐도 서초4동이 안정적으로 상위권을 지켰습니다(STEP6~7).

## STEP 11. 비즈니스 인사이트

접근성 있는 원자료를 "누구에게 어떤 카페를 열 것인가"라는 의사결정으로 옮기면 네 단계가 됩니다.

In [ ]:
최종3곳_요약 = 최종3곳[["행정동_코드_명", "점포_수", "카페1개당_평균매출_억원", "성장률", "균형점수"]].reset_index(drop=True)
최종3곳_요약.index = ["1순위", "2순위", "3순위"]
최종3곳_요약

**1. 최우선 후보 특정** — 지점을 한 곳만 낸다면 매출 규모가 가장 큰 **소공동**(1.61억원, 후보군 1위)을 우선 검토합니다. 다만 성장률이 9.6%로 세 후보 중 가장 낮아 "이미 성숙한 상권이라 지금이 정점에 가까울 수 있다"는 점은 함께 봐야 합니다. 성장성까지 함께 보고 싶다면 성장률 25.6%로 가장 높은 **잠실2동**(매출 2위)이 후보가 되지만, 점포 수 24개로 안정 표본 기준선(20개)을 겨우 넘겨 세 후보 중 표본이 가장 얇다는 한계가 있습니다.

**2. 컨셉·운영 설계** — 특정 시간대에 의존하지 않는 컨셉(하루 종일 고른 이용)을 원한다면 균형점수 94.0으로 아침·점심·저녁 매출이 가장 고르게 분산된 **서초4동**이 데이터상 가장 안전한 선택입니다. 반대로 소공동처럼 점심 시간대 비중이 큰(45.1%) 지역이라면, 점심 메뉴·회전율에 운영 역량을 집중하는 편이 그 지역의 실제 소비 패턴과 맞습니다.

**3. 하지 말아야 할 것(리스크)** — 후보를 볼 때 매출 스냅샷만 보고 판단하지 않습니다. 명동처럼 매출 상위권이면서도 최근 역성장 중인 곳이 있다는 걸 STEP3에서 확인했으므로, 성장률 필터를 기본으로 둡니다. 마찬가지로 컨셉_적합도(합계) 하나만 보고 판단하지도 않습니다 — 여의동처럼 합계 지표로는 1위지만 실제로는 특정 시간대(점심)에 쏠린 곳을 균형점수로 걸러냈습니다(STEP4~6). 마지막으로, 이 분석 결과를 "카페를 열면 매출이 오른다"는 식의 인과관계로 확대 해석하지 않습니다(STEP9).

**4. 검증 방법(다음 단계)** — 이 분석은 카페 업종의 매출·시간대·성장률 데이터만 사용했고, **임대료·권리금·경쟁 카페 밀도·상권 내 정확한 입지(1층/2층, 역과의 거리 등)는 포함하지 않았습니다.** 세 후보지 모두 실제 개업 결정 전에는 직접 답사와 임대 시세 확인이 필요하고, 표본이 작은 잠실2동은 특히 현장 확인의 비중을 높여야 합니다.

## STEP 12. 한계와 이를 넘으려면 필요한 데이터

지금의 한계는 대부분 분석 기법이 아니라 자료 자체에서 비롯되므로, 같은 데이터에 더 복잡한 모델을 얹는다고 풀리지 않습니다.

| 지금의 한계 | 필요한 데이터 | 해결되는 것 |
|---|---|---|
| 임대료·권리금·경쟁 카페 밀도가 데이터에 없음 | 상가 임대 시세, 동종업계 위치 데이터(예: 소상공인시장진흥공단 상권정보) | 매출뿐 아니라 비용까지 고려한 순이익 기준 입지 비교 |
| 상관관계이지 인과관계가 아님(단면 데이터) | 같은 상권을 여러 시점에 추적한 개업·폐업 패널 데이터 | 배후 수요 변화가 매출에 미치는 영향을 시간 순서로 확인(역인과 배제) |
| 잠실2동 표본 24개로 세 후보 중 가장 작음 | 인접 분기 데이터 추가 확보 또는 현장 실사 | 표본 안정성 보강, 우연한 결과일 위험 축소 |
| 행정동 데이터가 2025Q4까지라 자치구(2026Q1) 대비 약 1개 분기 시차 | 최신 분기 행정동 단위 데이터 | 가장 최근 트렌드까지 반영 |
| 컨셉_적합도·상권_변화_지표_명의 정확한 산정 기준 미확인 | 서울시 원본 지표 산출 방법론 문서 | 지표 해석의 정확도 향상 |

## 정리 — 프로젝트 마무리

- **최종 후보**: 소공동(매출 1위) · 잠실2동(매출 2위·최고 성장률) · 서초4동(균형점수 안정적 1위, 민감도 분석으로 검증).
- **방법론적 기여**: 컨셉_적합도(합계)가 시간대 쏠림을 못 잡아내는 한계를 확인하고, 균형점수(엔트로피)로 보완했습니다. 표본이 작은 곳에서 우연히 1위가 나올 위험은 민감도 분석(점포수 기준을 단계적으로 올려보기)으로 검증했습니다.
- **인과관계 아님을 명시**: 모델링 결과(직장인구·가구수·집객시설)는 상관관계이지 인과관계가 아니라는 점을 리포트에도 그대로 남깁니다.
- **데이터에 없는 것**: 임대료·권리금·경쟁 밀도는 이 프로젝트의 범위 밖이라 최종 의사결정 전 별도 확인이 필요합니다.

이 내용을 바탕으로 `README.md`와 최종 리포트에 문제정의 → EDA → 모델링 → 인사이트의 전체 흐름을 정리합니다.